WARRANTY SILVER LAYER CREATION

In [0]:
from pyspark.sql.functions import col, trim, upper

warranty_bronze = spark.table("bronze_warranty")

warranty_silver = warranty_bronze \
    .withColumn("sku", upper(trim(col("sku")))) \
    .withColumn("component_id", upper(trim(col("component_id"))))

display(warranty_silver)

sku,component_id,units_sold,failure_count,failure_rate
LAP1001,C001,750,206,0.2747
LAP1001,C002,750,29,0.0387
LAP1001,C003,750,132,0.176
LAP1001,C004,750,51,0.068
LAP1001,C005,750,44,0.0587
LAP1001,C006,750,66,0.088
LAP1002,C001,800,228,0.285
LAP1002,C002,800,33,0.0412
LAP1002,C003,800,147,0.1838
LAP1002,C004,800,57,0.0712


In [0]:
from pyspark.sql.functions import sum, when

warranty_silver.select(
    [
        sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
        for c in warranty_silver.columns
    ]
).show()

+---+------------+----------+-------------+------------+
|sku|component_id|units_sold|failure_count|failure_rate|
+---+------------+----------+-------------+------------+
|  0|           0|         0|            0|           0|
+---+------------+----------+-------------+------------+



In [0]:
warranty_silver.groupBy("sku", "component_id") \
    .count() \
    .filter(col("count") > 1) \
    .show()

+---+------------+-----+
|sku|component_id|count|
+---+------------+-----+
+---+------------+-----+



In [0]:
warranty_silver.filter(
    (col("units_sold") < 0) |
    (col("failure_count") < 0) |
    (col("failure_rate") < 0)
).show()

+---+------------+----------+-------------+------------+
|sku|component_id|units_sold|failure_count|failure_rate|
+---+------------+----------+-------------+------------+
+---+------------+----------+-------------+------------+



In [0]:
warranty_silver.filter(
    col("failure_rate") > 1
).show()

+---+------------+----------+-------------+------------+
|sku|component_id|units_sold|failure_count|failure_rate|
+---+------------+----------+-------------+------------+
+---+------------+----------+-------------+------------+



In [0]:
warranty_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_warranty")

In [0]:
display(spark.table("silver_warranty"))

sku,component_id,units_sold,failure_count,failure_rate
LAP1001,C001,750,206,0.2747
LAP1001,C002,750,29,0.0387
LAP1001,C003,750,132,0.176
LAP1001,C004,750,51,0.068
LAP1001,C005,750,44,0.0587
LAP1001,C006,750,66,0.088
LAP1002,C001,800,228,0.285
LAP1002,C002,800,33,0.0412
LAP1002,C003,800,147,0.1838
LAP1002,C004,800,57,0.0712


In [0]:
print(
    "Bronze Warranty records:",
    spark.table("bronze_warranty").count()
)

print(
    "Silver Warranty records:",
    spark.table("silver_warranty").count()
)

Bronze Warranty records: 60
Silver Warranty records: 60
